# MovieLens on OpenShift: Spark + Polaris + SeaweedFS

Follow the [MovieLens instructions](MovieLens.md) first. Use a fresh Python kernel in `spark-master-0`'s Jupyter container; run one notebook at a time.
This version uses the existing Spark 3.5.9 / Iceberg 1.10.0 image, Polaris `training` catalog and SeaweedFS `warehouse` bucket.
No Maven packages, pip installs, new container images or localhost service endpoints are required.

This version can prepare the real GroupLens `ml-latest-small` dataset, verify its checksum, and retain its README/usage terms. Read the linked usage terms before downloading; existing-data mode is also available for CSVs already stored in S3.
Bronze is never overwritten. Each kernel run uses isolated silver/gold namespaces; rerunning a write cell replaces only that run's output table.
See [MovieLens validation](MovieLens-VALIDATION.md) for execution results and SeaweedFS browser-access checks.


## Spark configuration

In [ ]:
import csv
import datetime
import hashlib
import io
import json
import os
from pathlib import Path
import re
import socket
import urllib.request
import uuid
import zipfile
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

RUN_ID = os.environ.get('MOVIELENS_RUN_ID') or ('movielens_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
if not re.fullmatch(r'[a-z][a-z0-9_]{0,90}', RUN_ID):
    raise ValueError('Use a simple lowercase MOVIELENS_RUN_ID')
POLARIS_URI = os.environ.get('POLARIS_URI', 'http://polaris:8181/api/catalog').rstrip('/')
S3_ENDPOINT = os.environ.get('S3_ENDPOINT', 'http://seaweedfs:8333')
key = os.environ.get('AWS_ACCESS_KEY_ID', 'team')
secret = os.environ.get('AWS_SECRET_ACCESS_KEY', 'team1234')
credential = os.environ.get('POLARIS_CLIENT_ID', 'team') + ':' + os.environ.get('POLARIS_CLIENT_SECRET', 'team1234')
driver_ip = os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname())
spark = (SparkSession.builder.appName('MovieLens-OpenShift-' + RUN_ID)
    .master('spark://spark-master:7077')
    .config('spark.driver.host', driver_ip).config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.cores.max', '1').config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m').config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.hadoop.fs.s3a.endpoint', S3_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', key).config('spark.hadoop.fs.s3a.secret.key', secret)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', POLARIS_URI)
    .config('spark.sql.catalog.polaris.warehouse', 'training')
    .config('spark.sql.catalog.polaris.oauth2-server-uri', POLARIS_URI + '/v1/oauth/tokens')
    .config('spark.sql.catalog.polaris.credential', credential)
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', S3_ENDPOINT)
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', key)
    .config('spark.sql.catalog.polaris.s3.secret-access-key', secret)
    .config('spark.sql.catalog.polaris.client.region', 'us-east-1')
    .config('spark.redaction.regex', '(?i)secret|password|token|access[.]?key|credential')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
SILVER = 'polaris.' + RUN_ID + '_silver'
GOLD = 'polaris.' + RUN_ID + '_gold'
print('Spark:', spark.version, 'master:', spark.sparkContext.master)
print('Run:', RUN_ID, 'silver:', SILVER, 'gold:', GOLD)


## Prepare real MovieLens bronze data, or select existing CSVs

In [ ]:
# Read GroupLens usage terms in README.md before downloading the dataset.
DATA_MODE = os.environ.get('MOVIELENS_DATA_MODE', 'download')  # 'download' or 'existing'
BRONZE = os.environ.get('MOVIELENS_BRONZE_PREFIX', 's3a://warehouse/movielens/bronze/ml-latest-small').rstrip('/')
MOVIES_PATH = BRONZE + '/movies/movies.csv'
RATINGS_PATH = BRONZE + '/ratings/ratings.csv'
DATASET_SHA256 = '696d65a3dfceac7c45750ad32df2c259311949efec81f0f144fdfb91ebc9e436'
source_movies = None
source_ratings = None
if DATA_MODE == 'download':
    archive_path = os.environ.get('MOVIELENS_ARCHIVE')
    if archive_path:
        archive = Path(archive_path).read_bytes()
    else:
        with urllib.request.urlopen('https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', timeout=60) as response:
            archive = response.read(10 * 1024 * 1024 + 1)
    if len(archive) > 10 * 1024 * 1024 or hashlib.sha256(archive).hexdigest() != DATASET_SHA256:
        raise RuntimeError('Dataset checksum/size differs from tested MovieLens small archive; inspect before proceeding')
    with zipfile.ZipFile(io.BytesIO(archive)) as z:
        # Select exact members, never extract arbitrary paths from an archive.
        movies_bytes = z.read('ml-latest-small/movies.csv')
        ratings_bytes = z.read('ml-latest-small/ratings.csv')
        source_readme = z.read('ml-latest-small/README.txt')
    source_movies = list(csv.DictReader(io.StringIO(movies_bytes.decode('utf-8'))))
    source_ratings = list(csv.DictReader(io.StringIO(ratings_bytes.decode('utf-8'))))
    jvm = spark.sparkContext._jvm
    conf = spark.sparkContext._jsc.hadoopConfiguration()
    def put_if_absent(uri, payload):
        path = jvm.org.apache.hadoop.fs.Path(uri)
        fs = path.getFileSystem(conf)
        if fs.exists(path):
            # Verify byte-for-byte before reusing bronze; never overwrite it.
            stream = fs.open(path)
            try:
                existing = jvm.org.apache.commons.io.IOUtils.toByteArray(stream)
                if hashlib.sha256(bytes(existing)).digest() != hashlib.sha256(payload).digest():
                    raise RuntimeError('Existing bronze differs; choose a new MOVIELENS_BRONZE_PREFIX: ' + uri)
            finally:
                stream.close()
            print('BRONZE_REUSED:', uri)
        else:
            stream = fs.create(path, False)
            try:
                stream.write(bytearray(payload))
            finally:
                stream.close()
            print('BRONZE_CREATED:', uri)
    put_if_absent(MOVIES_PATH, movies_bytes)
    put_if_absent(RATINGS_PATH, ratings_bytes)
    put_if_absent(BRONZE + '/README-MovieLens.txt', source_readme)
elif DATA_MODE != 'existing':
    raise ValueError('MOVIELENS_DATA_MODE must be download or existing')
print('Dataset mode:', DATA_MODE, 'bronze:', BRONZE)


## Read bronze CSVs from SeaweedFS through S3A

In [ ]:
movies_df = spark.read.option('header', 'true').option('mode', 'FAILFAST').schema(
    'movieId INT, title STRING, genres STRING').csv(MOVIES_PATH)
ratings_df = spark.read.option('header', 'true').option('mode', 'FAILFAST').schema(
    'userId INT, movieId INT, rating DOUBLE, timestamp LONG').csv(RATINGS_PATH)
movie_count, rating_count = movies_df.count(), ratings_df.count()
assert movie_count > 0 and rating_count > 0, 'Bronze CSVs must not be empty'
if DATA_MODE == 'download':
    assert (movie_count, rating_count) == (len(source_movies), len(source_ratings)) == (9742, 100836)
print('BRONZE_COUNTS:', movie_count, rating_count)
movies_df.show(5, truncate=False)
ratings_df.show(5, truncate=False)


## Isolated Polaris namespaces

In [ ]:
spark.sql('SHOW NAMESPACES IN polaris').show(truncate=False)
spark.sql(f'CREATE NAMESPACE IF NOT EXISTS {SILVER}')
spark.sql(f'CREATE NAMESPACE IF NOT EXISTS {GOLD}')


## Clean silver data with deterministic duplicate handling

In [ ]:
# Policy: choose lexical title/genres for duplicate movie IDs; highest rating
# for conflicting copies of the same user/movie/timestamp event. Adapt this
# explicitly for a different production ingestion policy.
movie_window = Window.partitionBy('movieId').orderBy(F.col('title'), F.col('genres').asc_nulls_last())
event_window = Window.partitionBy('userId', 'movieId', 'timestamp').orderBy(F.col('rating').desc())
silver_movies_df = (movies_df.filter('movieId IS NOT NULL AND movieId > 0 AND title IS NOT NULL')
    .withColumn('_rn', F.row_number().over(movie_window)).where('_rn = 1').drop('_rn'))
silver_ratings_df = (ratings_df.filter('userId IS NOT NULL AND userId > 0 AND movieId IS NOT NULL AND movieId > 0 AND timestamp IS NOT NULL AND timestamp >= 0')
    .filter(F.col('rating').between(0.5, 5.0) & ~F.isnan('rating'))
    .withColumn('_rn', F.row_number().over(event_window)).where('_rn = 1').drop('_rn'))


## Write silver Iceberg tables (only this run's namespace)

In [ ]:
silver_movies_df.writeTo(SILVER + '.movielens_movies').using('iceberg').createOrReplace()
silver_ratings_df.writeTo(SILVER + '.movielens_ratings').using('iceberg').createOrReplace()
assert spark.table(SILVER + '.movielens_movies').count() == silver_movies_df.count()
assert spark.table(SILVER + '.movielens_ratings').count() == silver_ratings_df.count()
spark.sql(f'SHOW TABLES IN {SILVER}').show(truncate=False)


## Gold analytics: at least 100 ratings and unrounded average at least 4.0

In [ ]:
gold_df = spark.sql(f'''SELECT m.movieId, m.title, m.genres, r.rating_count, r.avg_rating
 FROM {SILVER}.movielens_movies m JOIN (
  SELECT movieId, COUNT(*) AS rating_count, AVG(rating) AS avg_rating
  FROM {SILVER}.movielens_ratings GROUP BY movieId
  HAVING COUNT(*) >= 100 AND AVG(rating) >= 4.0
 ) r ON m.movieId = r.movieId''')
gold_df.writeTo(GOLD + '.movielens_top_movies').using('iceberg').createOrReplace()
spark.sql(f'''SELECT title, genres, rating_count, ROUND(avg_rating, 2) AS avg_rating
 FROM {GOLD}.movielens_top_movies
 ORDER BY avg_rating DESC, rating_count DESC, title LIMIT 20''').show(truncate=False)


## Verify gold against an independent source-CSV calculation

In [ ]:
gold_rows = spark.table(GOLD + '.movielens_top_movies').collect()
assert all(r.rating_count >= 100 and r.avg_rating >= 4.0 for r in gold_rows)
assert len({r.movieId for r in gold_rows}) == len(gold_rows)
if DATA_MODE == 'download':
    totals = {}
    for row in source_ratings:
        movie = int(row['movieId'])
        count, total = totals.get(movie, (0, 0.0))
        totals[movie] = (count + 1, total + float(row['rating']))
    oracle = {movie: (count, total / count) for movie, (count, total) in totals.items()
              if count >= 100 and total / count >= 4.0}
    assert {r.movieId for r in gold_rows} == set(oracle)
    assert all(r.rating_count == oracle[r.movieId][0] and abs(r.avg_rating - oracle[r.movieId][1]) < 1e-10 for r in gold_rows)
    assert len(gold_rows) > 0, 'Real small dataset should produce nonempty gold'
print('GOLD_ROWS:', len(gold_rows))
print('MOVIELENS_GOLD_VALIDATION_PASS')


## Inspect real Iceberg snapshots and physical S3 locations

In [ ]:
for table in (SILVER + '.movielens_movies', SILVER + '.movielens_ratings', GOLD + '.movielens_top_movies'):
    spark.sql(f'SELECT committed_at, snapshot_id, operation FROM {table}.snapshots ORDER BY committed_at DESC').show(truncate=False)
    files = spark.sql(f'SELECT file_path FROM {table}.files').collect()
    assert files and all(row.file_path.startswith(('s3://', 's3a://')) for row in files)
    print('ICEBERG_TABLE:', table, 'DATA_FILE:', files[0].file_path)


## Finish

In [ ]:
spark.stop()
print('MOVIELENS_NOTEBOOK_PASS')
